<a href="https://colab.research.google.com/github/nterreri-cmu/p2-datacenters/blob/main/clean_datacenters.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [87]:
# import Pandas
import pandas as pd

# import numpy
import numpy as np

#matplot
import matplotlib.pyplot as plt

#seaborn
import seaborn as sns

#api calls
import requests # to download
import duckdb # to query

#finds numbers in text
import re




In [88]:
#Upload data center data file

dc_url = 'https://raw.githubusercontent.com/nterreri-cmu/p2-datacenters/refs/heads/main/Data_Centers_Database_Copy%20-%20FracTracker%20Data%20Centers.csv'

datacenters_df = pd.read_csv(dc_url)


In [89]:
#EDA- 1,701 rows and 44 columns
print(datacenters_df.shape)

(1701, 44)


In [90]:
#EDA Columns
print(datacenters_df.columns.tolist())


['facility_name', 'address', 'city', 'state', 'zip', 'county', 'lat', 'long', 'status', 'location_confidence', 'purpose', 'operator_name', 'tenant', 'mw', 'sizerank', 'power_source', 'dedicated_power_plant', 'number_of_generators', 'number_of_buildings', 'cooling_source', 'cooling_type', 'facility_size_sqft', 'property_size_acres', 'project_cost', 'expected_date_online', 'community_pushback', 'advocacy_information', 'resistance_status', 'nda', 'community_group_website_1', 'community_group_website_2', 'petition_url', 'other_info', 'information_source', 'info_source_1', 'info_source_2', 'info_source_3', 'info_source_4', 'info_source_5', 'info_source_6', 'info_source_7', 'info_source_8', 'date_created', 'date_updated']


In [91]:
datacenters_df = datacenters_df.dropna(how="all")                 # drop blank rows at the bottom
datacenters_df = datacenters_df.dropna(axis=1, how="all")         # drop blank trailing columns
datacenters_df = datacenters_df.loc[:, ~datacenters_df.columns.str.startswith("Unnamed")] #drop empty columns

In [92]:
#keeps columns with enough data
keep = ["facility_name", "address", "city", "state", "zip", "county", "lat", "long",
        "status", "location_confidence", "operator_name", "mw", "sizerank",
        "facility_size_sqft", "property_size_acres", "community_pushback",
        "information_source", "date_created", "date_updated"]

#rename long to lon
datacenters_df = datacenters_df[keep].rename(columns={"long": "lon"})

In [93]:
#removes any extra spaces and make everything a capital letter
datacenters_df["state"] = datacenters_df["state"].astype(str).str.strip().str.upper()

#this data center was placed in the wrong state, fixed it
datacenters_df.loc[datacenters_df["facility_name"].str.contains("LFF Industrial", na=False), "state"] = "IL"

print(sorted(datacenters_df["state"].unique()))

['AK', 'AL', 'AR', 'AZ', 'CA', 'CO', 'CT', 'DE', 'FL', 'GA', 'HI', 'IA', 'ID', 'IL', 'IN', 'KS', 'KY', 'LA', 'MA', 'MD', 'ME', 'MI', 'MN', 'MO', 'MS', 'MT', 'NC', 'ND', 'NE', 'NH', 'NJ', 'NM', 'NV', 'NY', 'OH', 'OK', 'OR', 'PA', 'SC', 'SD', 'TN', 'TX', 'UT', 'VA', 'WA', 'WI', 'WV', 'WY']


In [94]:
#drop the data points that explicitly say they are not a data center
before = len(datacenters_df)
datacenters_df = datacenters_df[~datacenters_df["facility_name"].str.contains("not data center", case=False, na=False)]




In [95]:
#Inconsistent capitalization in columns

for c in ["community_pushback", "location_confidence"]:
    datacenters_df_clean[c] = datacenters_df_clean[c].str.strip().str.capitalize()

In [96]:
#rename names that are a weblink web link (starts with "http") and give it a readable name.
url_name = datacenters_df["facility_name"].str.startswith("http", na=False)
datacenters_df.loc[url_name, "facility_name"] = "Prologis Shelbyville Data Center"
datacenters_df[url_name][["facility_name", "city", "state"]]

,facility_name,city,state
414,Prologis Shelbyville Data Center,Shelbyville,IN


In [97]:
#grouping status into 3 useful groups
status_map = {
    "Operating": "operating",
    "Expanding": "operating",
    "Proposed": "planned",
    "Pre-proposal": "planned",
    "Approved/Permitted/Under construction": "planned",
    "Suspended": "stalled",
    "Cancelled": "stalled",
}
datacenters_df["status"] = datacenters_df["status"].astype(str).str.strip()
datacenters_df["status_group"] = datacenters_df["status"].map(status_map)

print(datacenters_df.loc[datacenters_df["status_group"].isna(), "status"].value_counts())
pd.crosstab(datacenters_df["status"], datacenters_df["status_group"])

Series([], Name: count, dtype: int64)


status_group,operating,planned,stalled
status,,,
Approved/Permitted/Under construction,0,178,0
Cancelled,0,0,78
Expanding,69,0,0
Operating,534,0,0
Pre-proposal,0,11,0
Proposed,0,758,0
Suspended,0,0,72


In [98]:
#standardizes Mega Campus string
datacenters_df_clean["sizerank"] = datacenters_df_clean["sizerank"].str.strip().replace({
    "Mega Campus (>1,000 MW)": "Mega campus (>1,000 MW)",
    "Unknown": np.nan})
#Put size rank in order
size_order = ["Small (0-10 MW)", "Medium (11-50 MW)", "Large (51-99 MW)",
              "Hyperscale (100-999 MW)", "Mega campus (>1,000 MW)"]
datacenters_df["sizerank"] = datacenters_df["sizerank"].astype(str).str.strip().replace({"Unknown": np.nan, "nan": np.nan})
datacenters_df["sizerank"] = pd.Categorical(datacenters_df["sizerank"], categories=size_order, ordered=True)
datacenters_df["sizerank"].value_counts(dropna=False)

,count
sizerank,
NaN,887
Hyperscale (100-999 MW),395
Medium (11-50 MW),139
"Mega campus (>1,000 MW)",122
Small (0-10 MW),116
Large (51-99 MW),41


In [99]:
print(datacenters_df.shape)
print(datacenters_df["state"].value_counts().head(10))



(1700, 20)
state
VA    464
TX    220
GA    183
PA    108
OH     66
CA     44
IN     43
IL     41
NY     35
MO     34
Name: count, dtype: int64


In [100]:
datacenters_df_clean = datacenters_df
datacenters_df_clean.to_csv("datacenters_clean.csv", index=False)
